In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import matplotlib.pyplot as plt

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
def process_lead_years(exp_ctrl, exp_sens, var, y1, y2, save_path):
    lead = f"{y1}-{y2}"
    logging.info(f"Starting main {lead}")
    try:
        dset_ctrl = xr.open_mfdataset(f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias.nc')
        logging.info(f"{dset_ctrl}")
        dset_sens = xr.open_mfdataset(f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias.nc')
        
        dset_ctrl_p = xr.open_mfdataset(f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias_p.nc')
        
        dset_sens_p = xr.open_mfdataset(f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias_p.nc')
    
        levels=[-0.5, -0.4, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5]
        title = f'a) DCPP-CTRL'
        af.map_plot(dset_ctrl[var], dset_ctrl_p, levels=levels, title=title, cmap='BrBG', antartica=False)
        plt.savefig(f'{FIG_DIR}/{exp_ctrl}_{var}_bias.png', dpi=600, bbox_inches = 'tight')

        title = f'b) DCPP-SENS'
        af.map_plot(dset_sens[var], dset_sens_p, levels=levels, title=title, cmap='BrBG', antartica=False)
        plt.savefig(f'{FIG_DIR}/{exp_sens}_{var}_bias.png', dpi=600, bbox_inches = 'tight')

        
        
        logging.info(f"Bias files written successfully for lead {lead}")
    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
%%time
def main(exp_ctrl, exp_sens, var, SAVE_PATH):
    """Funzione principale per elaborare i dati in parallelo."""
    futures = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        for y1 in range(5):
            for y2 in range(y1 + 1, 5):
                logging.info(f"Submitting task for lead years {y1}-{y2}")
                futures.append(
                    executor.submit(process_lead_years, exp_ctrl, exp_sens, var, y1, y2, SAVE_PATH)
                )
        # Wait for all tasks to complete
        # progresso: stampa ogni task man mano che termina
        for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
            _f.result()
            print(f"  completato {_i}/{len(futures)}", flush=True)

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    
    # Define or import exp_ctrl, exp_sens, var, and process_lead_years
    main(exp_ctrl, exp_sens, var, SAVE_PATH)